In [ ]:
import pandas as pd
import warnings
import numpy as np
import os
import glob
from datetime import datetime

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
          

folder_path =r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Kapture_Reports\Kapture_Email\Email_Inbound"
excel_files = glob.glob(os.path.join(folder_path,"*.xlsx"))

required_columns = ["Ticket No", "Source", "Created Date", "Created Time", "Interation count Customer", "Queue"]
df_list = []

for file in excel_files:
    try:
        df= pd.read_excel(file, engine = "openpyxl", usecols = required_columns)
        df_list.append(df)
    except Exception as e:
        print(f"Error Reading {file}: {e}")
        
combined_df = pd.concat(df_list, ignore_index = True) if df_list else pd.DataFrame()

file_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Python_Email\Plan_Name.xlsx"
df_new = pd.read_excel(file_path)

Merged_df = pd.merge(combined_df,df_new, on="Queue", how ="inner")

Merged_df["Created Date Clean"] = pd.to_datetime(Merged_df["Created Date"], errors = "coerce").dt.date
Merged_df["Created Time Clean"] = pd.to_datetime(Merged_df["Created Time"].astype(str), format ="%H:%M:%S", errors="coerce").dt.time

Merged_df["Created_DateTime"] = pd.to_datetime(
    Merged_df["Created Date Clean"].astype(str) +' '+Merged_df["Created Time Clean"].astype(str, errors= "coerce")
)

Merged_df["Time_30min"] = Merged_df["Created_DateTime"].dt.floor("30min").dt.time
print(Merged_df.head(10))

In [ ]:
print(Merged_df.columns)
filtered_df = Merged_df.copy()
null_counts = filtered_df.isnull().sum()
print(" <Null value Count>\n\n",null_counts)

data_types = filtered_df.dtypes
#print(" data types \n\n",data_types)
print(filtered_df["Ticket No"].isnull().sum())

In [ ]:
filtered_df = filtered_df.dropna(subset=['Process','Source'])
                             
filtered_df = filtered_df[
  (filtered_df["Process"] == "PL Email") &
  (filtered_df['Source'] =="Email")]
print(filtered_df)

In [32]:
from datetime import datetime, timedelta

# Step 1: Generate all 30-min time slots in 24 hours
time_slots = []
t = datetime.strptime("00:00", "%H:%M")
end = datetime.strptime("23:30", "%H:%M")
while t <= end:
    time_slots.append(t.time())
    t += timedelta(minutes=30)
filtered_df["Date"] = pd.to_datetime(filtered_df["Date"]).dt.date

pivot_table = filtered_df.pivot_table(
    index="Time_30min",
    columns="Date",
    values="Ticket No",
    aggfunc="count",
    fill_value=0
)

pivot_table = pivot_table.reindex(index=time_slots, fill_value=0)

pivot_table.columns = [col.strftime("%Y-%m-%d") for col in pd.to_datetime(pivot_table.columns)]
import pandas as pd
pd.set_option("display.max_columns", None)
print(pivot_table)
print(pivot_table.columns)




            1900-01-01
Time_30min            
00:00:00           217
00:30:00           290
01:00:00           113
01:30:00            82
02:00:00            60
02:30:00            81
03:00:00            31
03:30:00            56
04:00:00            31
04:30:00            38
05:00:00            56
05:30:00            57
06:00:00            72
06:30:00            95
07:00:00           154
07:30:00           271
08:00:00           328
08:30:00           444
09:00:00          1135
09:30:00          1193
10:00:00          1900
10:30:00          2079
11:00:00          2117
11:30:00          2378
12:00:00          2478
12:30:00          2389
13:00:00          1904
13:30:00          2140
14:00:00          2068
14:30:00          2261
15:00:00          2197
15:30:00          2417
16:00:00          2118
16:30:00          2267
17:00:00          1811
17:30:00          1693
18:00:00          1197
18:30:00           813
19:00:00           677
19:30:00           745
20:00:00           606
20:30:00   

In [11]:
import pandas as pd

output_file = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Python_Email\Interval_Kapture_output.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    pivot_table.to_excel(writer, sheet_name="Pivot_Summary")

print(f"Pivot table saved to: {output_file}")


Pivot table saved to: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Python_Email\Interval_Kapture_output.xlsx


In [ ]:
print(Merged_df[["Created Date Clean", "Created Time Clean"]].head())
